# Day 21: Qdrant Performance Benchmarking
## Measuring Search Latency Across Scales

Welcome to Day 21 of the 90-Day AI Engineering Mastery curriculum. Today, we focus on a critical production engineering skill: performance benchmarking. We will measure how search latency in Qdrant scales when we move from 1,000 to 10,000 vectors.

### Core Theory (Just-in-Time)

**Why Benchmark?**
In production, vector databases like Qdrant need to handle rapid similarity searches across millions of vectors. While HNSW (Hierarchical Navigable Small World) graphs — the underlying indexing algorithm in Qdrant — offer sub-linear search time (typically `O(log N)`), latency still grows as the dataset size increases. 

**How does Qdrant handle this?**
Qdrant builds an HNSW index to quickly approximate nearest neighbors. The time it takes to search depends on:
1. **Dataset Size (N):** The number of vectors in the collection.
2. **Vector Dimensionality (D):** The size of each vector (e.g., 1536 for OpenAI embeddings).
3. **Search Parameters (`hnsw_ef`, `limit`):** Higher `limit` (top K results) or `hnsw_ef` (search accuracy parameter) increases latency.

Today, we will set up an isolated benchmark using Qdrant's `:memory:` mode to see the latency difference between a 1,000-vector collection and a 10,000-vector collection firsthand.

### Common Pitfalls in Production
1. **Ignoring Indexing Time:** HNSW indexes take time to build. In this benchmark, we focus on *search* latency, but inserting 10,000 vectors might take noticeably longer than 1,000.
2. **Over-provisioning Dimensions:** Higher dimensions increase memory usage and distance calculation time. Only use the dimensionality you actually need.
3. **Benchmarking with In-Memory vs. Disk:** Memory mode (`:memory:`) is fast but doesn't reflect the I/O bottleneck you might hit when vectors are stored on disk in a real cluster.


In [1]:
import time
import uuid
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# Initialize a local in-memory Qdrant client
client = QdrantClient(":memory:")

# Benchmark Parameters
DIMENSION = 128  # Using a smaller dimension for faster local generation
COLLECTION_1K = "benchmark_1k"
COLLECTION_10K = "benchmark_10k"

# 1. Create Collections
def setup_collection(collection_name: str) -> None:
    """
    Sets up a Qdrant collection with the specified name and vector parameters.
    If the collection already exists, it will be recreated.
    
    Args:
        collection_name (str): The name of the collection to setup.
    """
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)
        
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=DIMENSION, distance=Distance.COSINE),
    )

setup_collection(COLLECTION_1K)
setup_collection(COLLECTION_10K)

# 2. Generate and Insert Data
def generate_points(num_points: int) -> list[PointStruct]:
    """
    Generates a list of random point structures for insertion into Qdrant.
    
    Args:
        num_points (int): The number of points to generate.
        
    Returns:
        list[PointStruct]: A list of generated points with random vectors.
    """
    points = []
    for i in range(num_points):
        # Generate random vectors
        vector = np.random.rand(DIMENSION).astype(np.float32).tolist()
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={"index": i}
            )
        )
    return points

print("Generating and uploading 1,000 vectors...")
points_1k = generate_points(1000)
client.upload_points(collection_name=COLLECTION_1K, points=points_1k)

print("Generating and uploading 10,000 vectors...")
points_10k = generate_points(10000)
# Upload in batches to avoid overwhelming the memory client
client.upload_points(collection_name=COLLECTION_10K, points=points_10k)

# 3. Perform the Benchmark
def benchmark_search(collection_name: str, num_queries: int = 100) -> float:
    """
    Benchmarks the average search latency for a given collection.
    
    Args:
        collection_name (str): The name of the collection to benchmark.
        num_queries (int): The number of search queries to perform.
        
    Returns:
        float: The average search latency in milliseconds.
    """
    total_time = 0.0
    for _ in range(num_queries):
        query_vector = np.random.rand(DIMENSION).astype(np.float32).tolist()
        
        start_time = time.perf_counter()
        client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=10  # Top 10 results
        )
        end_time = time.perf_counter()
        
        total_time += (end_time - start_time)
        
    avg_latency_ms = (total_time / num_queries) * 1000
    return avg_latency_ms

print("\n--- Benchmark Results ---")
latency_1k = benchmark_search(COLLECTION_1K)
print(f"Average Search Latency (1,000 vectors): {latency_1k:.4f} ms")

latency_10k = benchmark_search(COLLECTION_10K)
print(f"Average Search Latency (10,000 vectors): {latency_10k:.4f} ms")
print("-------------------------")


Generating and uploading 1,000 vectors...
Generating and uploading 10,000 vectors...



--- Benchmark Results ---
Average Search Latency (1,000 vectors): 1.0271 ms


Average Search Latency (10,000 vectors): 5.3718 ms
-------------------------


### Practical Lab / Homework

**Task:** 
Now that you have seen how collection size affects latency, your task is to measure how the `limit` parameter (the number of requested nearest neighbors) impacts search time.

Using the `benchmark_10k` collection created above, write a function that benchmarks the average search latency for different values of `limit`: `[10, 50, 100, 500]`. 
Print the average latency for each limit.

**Reference Implementation:**
*(Try to write it yourself first, then review the implementation below)*


In [2]:
def benchmark_limit(collection_name: str, limits: list[int], num_queries: int = 50) -> None:
    """
    Benchmarks search latency across different limit values (top-k).
    
    Args:
        collection_name (str): The name of the collection to search.
        limits (list[int]): A list of limit values to benchmark.
        num_queries (int): The number of queries to average over.
    """
    print(f"Benchmarking limits on {collection_name}:")
    for limit in limits:
        total_time = 0.0
        for _ in range(num_queries):
            query_vector = np.random.rand(DIMENSION).astype(np.float32).tolist()
            
            start_time = time.perf_counter()
            client.query_points(
                collection_name=collection_name,
                query=query_vector,
                limit=limit
            )
            end_time = time.perf_counter()
            
            total_time += (end_time - start_time)
            
        avg_latency_ms = (total_time / num_queries) * 1000
        print(f"  Limit = {limit:<4} | Avg Latency: {avg_latency_ms:.4f} ms")

# Execute the lab benchmark
limits_to_test = [10, 50, 100, 500]
benchmark_limit(COLLECTION_10K, limits_to_test)


Benchmarking limits on benchmark_10k:


  Limit = 10   | Avg Latency: 4.9355 ms


  Limit = 50   | Avg Latency: 5.6811 ms


  Limit = 100  | Avg Latency: 6.9400 ms


  Limit = 500  | Avg Latency: 17.0198 ms
